In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.model_selection import train_test_split
# from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor

from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import pyarrow


# import warnings
# warnings.filterwarnings("ignore")


In [ ]:

#### We load the first 200000 rows only 
data = pd.read_csv('../data/train.csv', nrows=200000)
print (data.shape)
data.head()

(200000, 6)


,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN


In [9]:
data.describe()

,id,store_nbr,item_nbr,unit_sales,onpromotion
count,200000.000000,200000.000000,2.000000e+05,200000.000000,0.0
mean,99999.500000,25.786580,6.153045e+05,10.430063,NaN
std,57735.171256,16.217806,3.029205e+05,18.617576,NaN
min,0.000000,1.000000,9.699500e+04,-27.000000,NaN
25%,49999.750000,10.000000,3.579620e+05,2.000000,NaN
50%,99999.500000,26.000000,5.869690e+05,5.000000,NaN
75%,149999.250000,41.000000,8.723080e+05,11.000000,NaN
max,199999.000000,54.000000,1.118683e+06,979.000000,NaN


In [ ]:
####Run this only once to change the saving data

## Skipping the id column and changing the datatypes

df = pd.read_csv(
    '../data/train.csv',
    usecols=["date", "store_nbr", "item_nbr", "unit_sales", "onpromotion"],
    dtype={
        # "id": "int32",
        "store_nbr": "int8",
        "item_nbr": "int32",
        "unit_sales": "float32",
        "onpromotion": "boolean",
    },
    parse_dates=["date"],
)


###Saving into parquet
df.to_parquet('train.parquet', index = False)

In [2]:
df = pd.read_parquet('train.parquet')

In [3]:
df.describe()

,date,store_nbr,item_nbr,unit_sales
count,125497040,1.254970e+08,1.254970e+08,1.254970e+08
mean,2015-09-15 16:43:18.789250,2.746458e+01,9.727692e+05,8.554866e+00
min,2013-01-01 00:00:00,1.000000e+00,9.699500e+04,-1.537200e+04
25%,2014-09-20 00:00:00,1.200000e+01,5.223830e+05,2.000000e+00
50%,2015-11-21 00:00:00,2.800000e+01,9.595000e+05,4.000000e+00
75%,2016-10-17 00:00:00,4.300000e+01,1.354380e+06,9.000000e+00
max,2017-08-15 00:00:00,5.400000e+01,2.127114e+06,8.944000e+04
std,NaN,1.633051e+01,5.205336e+05,2.360515e+01


In [4]:
# checking to see how many null values we have in each of the other columns
columns_with_nulls = []
for column in df.columns:
    if df.loc[:, column].isnull().any():
        columns_with_nulls.append(column)

for column in columns_with_nulls:
    percent_missing = round(((df.loc[df.loc[:, column].isnull()].shape[0] / df.shape[0]) * 100), 4)
    print("Column {} has {}% missing values \n".format(column, percent_missing))

Column onpromotion has 17.2575% missing values 



In [5]:
df["promo_missing"] = df["onpromotion"].isna()

# date range of missing vs. present
df.groupby("promo_missing")["date"].agg(["min", "max", "count"])

# share missing per month
df.groupby(df["date"].dt.to_period("M"))["promo_missing"].mean()

date
2013-01    1.0
2013-02    1.0
2013-03    1.0
2013-04    1.0
2013-05    1.0
2013-06    1.0
2013-07    1.0
2013-08    1.0
2013-09    1.0
2013-10    1.0
2013-11    1.0
2013-12    1.0
2014-01    1.0
2014-02    1.0
2014-03    1.0
2014-04    0.0
2014-05    0.0
2014-06    0.0
2014-07    0.0
2014-08    0.0
2014-09    0.0
2014-10    0.0
2014-11    0.0
2014-12    0.0
2015-01    0.0
2015-02    0.0
2015-03    0.0
2015-04    0.0
2015-05    0.0
2015-06    0.0
2015-07    0.0
2015-08    0.0
2015-09    0.0
2015-10    0.0
2015-11    0.0
2015-12    0.0
2016-01    0.0
2016-02    0.0
2016-03    0.0
2016-04    0.0
2016-05    0.0
2016-06    0.0
2016-07    0.0
2016-08    0.0
2016-09    0.0
2016-10    0.0
2016-11    0.0
2016-12    0.0
2017-01    0.0
2017-02    0.0
2017-03    0.0
2017-04    0.0
2017-05    0.0
2017-06    0.0
2017-07    0.0
2017-08    0.0
Freq: M, Name: promo_missing, dtype: float64

In [6]:
first_valid_date = df.loc[df["onpromotion"].notna(), "date"].min()
print(first_valid_date)

#### Checking that there is no missing item after this date
after = df[df["date"] >= first_valid_date]
print(after["onpromotion"].isna().sum())   # should be 0

2014-04-01 00:00:00
0


In [7]:
### Droping the old missing data
df = df[df["date"] >= first_valid_date]
df.shape

(103839389, 6)

In [ ]:

# splitting data into features and targets 
X_train= df.drop(columns='unit_sales')
y_train = df.loc[:, 'unit_sales']





def add_date_features(df):
    df = df.copy()
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    return df.drop(columns="date")

X_train = add_date_features(X_train)
X_train.to_parquet('X_train.parquet')


In [ ]:
X_test = pd.read_csv('../data/test.csv')
X_test = add_date_features(X_test)
X_test.to_parquet('X_test.parquet')

In [ ]:
# # #### saving the data to parquet



# #loading the data
# X_train = pd.read_csv(
#     "X.csv",
#     dtype={
#         "store_nbr": "int8",
#         "item_nbr": "int32",
#         "onpromotion": "bool",   # or "bool" if it's still True/False
#     },
#     parse_dates=["date"],
# )

# y_train = pd.read_csv("y.csv", dtype="float32")
# y_train.to_parquet('y_train.parquet', index = False)



# # X_train["year"] = X_train["date"].dt.year
# # X_train["month"] = X_train["date"].dt.month
# # X_train["day"] = X_train["date"].dt.day
# # X_train["dayofweek"] = X_train["date"].dt.dayofweek
# # X_train = X_train.drop(columns="date")
# def add_date_features(df):
#     df = df.copy()
#     df["year"] = df["date"].dt.year
#     df["month"] = df["date"].dt.month
#     df["day"] = df["date"].dt.day
#     df["dayofweek"] = df["date"].dt.dayofweek
#     return df.drop(columns="date")

# X_train = add_date_features(X_train)


# ### only once 
# X_train.to_parquet("X_train.parquet", index=False)




# #loading the data
# X_test= pd.read_csv(
#     "../data/test.csv",

#     dtype={
#         "store_nbr": "int8",
#         "item_nbr": "int32",
#         "onpromotion": "bool",   # or "bool" if it's still True/False
#     },
#     parse_dates=["date"],
# )

# X_test = add_date_features(X_test)
# ### only once 
# X_test.to_parquet("X_test.parquet", index=False)

# # y_test = pd.read_csv("test.csv", usecols='sal' dtype="float32").squeeze("columns")






/tmp/ipykernel_58519/1816365686.py:31: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["dayofweek"] = df["date"].dt.dayofweek
/tmp/ipykernel_58519/1816365686.py:31: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["dayofweek"] = df["date"].dt.dayofweek


In [4]:
print(X_train.shape)
print (X_train.head())

(103839389, 7)
   store_nbr  item_nbr  onpromotion  promo_missing  month  day  dayofweek
0          1    103520        False          False      4    1          1
1          1    103665        False          False      4    1          1
2          1    105574        False          False      4    1          1
3          1    105575        False          False      4    1          1
4          1    105577        False          False      4    1          1


In [6]:
##### Rading the parquet files
X_train = pd.read_parquet('X_train.parquet')
y_train = pd.read_parquet('y_train.parquet')
X_test = pd.read_parquet('X_test.parquet')

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)


# Preprocessing of training data, fit model 
rf_model.fit(X_train, y_train)

# Preprocessing of test data, get predictions
preds = X_test['id']
preds.stack (rf_model.predict(X_test.drop(columns='id')))

preds.to_csv('preds.csv', index =False)

# # Evaluate the model
# score = mean_absolute_error(y_test, preds)
# print('MAE for Random Forest', score)

In [8]:
sample = pd.read_csv('../data/sample_submission.csv')

In [11]:
sample.columns

Index(['id', 'unit_sales'], dtype='str')

In [12]:
sample.head(200)

,id,unit_sales
0,125497040,0
1,125497041,0
2,125497042,0
3,125497043,0
4,125497044,0
...,...,...
195,125497235,0
196,125497236,0
197,125497237,0
198,125497238,0


In [13]:
X_test.head()

,id,store_nbr,item_nbr,onpromotion,year,month,day,dayofweek
0,125497040,1,96995,False,2017,8,16,2
1,125497041,1,99197,False,2017,8,16,2
2,125497042,1,103501,False,2017,8,16,2
3,125497043,1,103520,False,2017,8,16,2
4,125497044,1,103665,False,2017,8,16,2
